[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter04_activation_functions_and_nonlinearity/notebook.ipynb)

# Chapter 4: Activation Functions and Nonlinearity

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

For most of the 2000s, deep neural networks had a reputation for being almost impossible to train. A network with one or two hidden layers could learn well enough, but adding more layers, which was supposed to make a network more powerful, usually made it worse. The early layers seemed to learn painfully slowly or not at all, and many researchers quietly went back to other methods. In 2006 Geoffrey Hinton and his colleagues found one way around the problem, which was to prepare each layer separately with a long warm up step before the real training began. It worked, but it was slow and fiddly, and nobody was quite sure why it was needed.

Then the answer turned out to be much simpler than anyone expected. In 2010 Vinod Nair and Geoffrey Hinton tried a very plain kind of bend inside each neuron, one that keeps positive numbers exactly as they are and turns every negative number into zero. A year later, Xavier Glorot, Antoine Bordes and Yoshua Bengio at the University of Montreal showed that deep networks built with this bend could reach their best performance with ordinary training alone, with no special warm up at all. The bend was called the **rectified linear unit**, or **ReLU** for short.

When AlexNet won the ImageNet contest in 2012, as you read in Chapter 1, ReLU was one of its key ingredients. Its authors reported that a network using ReLU reached a set level of accuracy on a standard image collection six times faster than the same network using the bend that had been popular until then. Today ReLU, or a close relative of it, sits inside almost every deep network you will ever use.

This chapter is about that bend inside every neuron, which is called the **activation function**. It starts with a surprise, which is that a network with many layers but no bends at all is no more powerful than a single neuron. Before you read on, try to guess the answers to these four questions.

1. **If you stack ten layers that only multiply and add, what do you get?**
2. **What could make the early layers of a deep network learn so much more slowly than the late ones?**
3. **Why would a bend as plain as "turn negatives into zero" work better than a smooth curve?**
4. **Should every layer of a network use the same activation, including the last one?**

## Why a Network Needs a Bend

Each layer you built in Chapter 3 did two things. It multiplied its inputs by a weight matrix and added a bias, and then it passed the result through an activation. The first part on its own is called a **linear** step, because it can only ever stretch, turn and shift numbers along straight lines. It is natural to wonder whether the activation is really needed, or whether stacking enough linear steps would do the job by itself.

Think of a photocopier with a zoom button. If you shrink a page to 50 percent and then enlarge the copy to 300 percent, the final page is simply 150 percent of the original. Two zooms in a row are just one zoom, and a hundred zooms in a row would still be just one zoom. Linear layers behave exactly the same way. The next cells build two linear layers with no activation between them and pass one input through both.

In [ ]:
import numpy as np

x = np.array([2.0, 1.0])
W1 = np.array([[1.0, -2.0],
               [0.5,  1.0],
               [3.0,  0.0]])
b1 = np.array([0.5, -1.0, 2.0])
W2 = np.array([[1.0, 2.0, -1.0]])
b2 = np.array([0.5])

In [ ]:
print(W2 @ (W1 @ x + b1) + b2)

Now combine the two layers into one before you start. Multiplying the two weight matrices together gives a single matrix, and pushing the first bias through the second layer gives a single bias. The combined layer gives exactly the same answer.

In [ ]:
W = W2 @ W1
b = W2 @ b1 + b2
print(W, b)
print(W @ x + b)

The two layers, with their nine weights and four biases, were secretly one layer with two weights and one bias all along, and the same would be true for any input you tried. Figure 4.1 shows the idea. **Without an activation between them, any number of linear layers collapse into a single linear layer, so a deep network with no bends is no more powerful than one neuron.**

> 🖼️ **Figure 4.1:** Two linear layers with no activation between them give exactly the same answer as one combined layer, just as two photocopier zooms make one zoom.

You can see the same thing on a real problem. The next cells bring back the plants in the field from Chapter 1, where the healthy variety grows in a circle surrounded by a ring of the weaker one. The network now has three hidden layers of eight neurons each, and the cell trains it twice, once with no activation at all, which scikit learn calls `"identity"`, and once with ReLU.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X, y = make_circles(n_samples=400, noise=0.08,
                    factor=0.45, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7)

In [ ]:
for act in ["identity", "relu"]:
    net = MLPClassifier(hidden_layer_sizes=(8, 8, 8),
                        activation=act, max_iter=3000,
                        random_state=7)
    net.fit(X_train, y_train)
    print(act, round(net.score(X_test, y_test), 2))

With no bends, three hidden layers scored 0.53, about the same as the single straight line in Chapter 1, because three straight steps are just one straight step. With ReLU the same network scored a perfect 1.0. The bend is what lets each layer reshape the space the way the XOR network did in Chapter 3, folding it so that the next layer has an easier job. A bend of this kind is called a **nonlinearity**, since it is not a straight line.

## Sigmoid and Tanh

You met the **sigmoid** in Chapter 2 as a dimmer switch, a smooth S shaped curve between 0 and 1, and for decades it was the natural choice for every neuron. Its close cousin, the **hyperbolic tangent** or **tanh**, has the same S shape but stretches from $-1$ to 1 instead, so its outputs are centred on zero.

For learning, what matters most is the curve's **slope**, meaning how much of a small change in the weighted sum gets through to the output, since Chapter 2 showed that learning feeds on small changes. The next cell computes the sigmoid and its slope at five points.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.array([-6, -2, 0, 2, 6])
s = sigmoid(z)
print(s.round(3))
print((s * (1 - s)).round(3))

The second line is the slope, and it has a surprisingly simple formula, the sigmoid's output times one minus that output. Two facts stand out. The slope is largest in the middle, but even there it is only 0.25, so a neuron passes on at most a quarter of any small change. Far from the middle, at $-6$ or 6, the curve is almost flat and the slope is only 0.002. A neuron sitting out there is **saturated**, which means it has stopped responding to changes in its input. The tanh behaves a little better, as the next cell shows.

In [ ]:
t = np.tanh(z)
print(t.round(3))
print((1 - t ** 2).round(3))

The tanh's slope reaches a full 1 in the middle, four times the sigmoid's best, but it also flattens out completely at the edges. Figure 4.2 draws both curves and their slopes, and the flat edges are where learning goes quiet.

> 🖼️ **Figure 4.2:** Sigmoid and tanh on the left and their slopes on the right. Both slopes fade to almost nothing once the weighted sum moves far from zero, and the sigmoid's slope never rises above 0.25.

A small slope in one neuron is not a disaster. The trouble starts in a deep network, because in Chapter 7 you will see that the signal used for learning has to travel backwards through every layer, and at each sigmoid layer it is multiplied by that layer's slope. It is like a message passed down a long line of people, where each person can only repeat it at a quarter of the volume they heard. The next cell shows how much of the signal is left, in percent, after one to six sigmoid layers, even in the best case where every slope is 0.25.

In [ ]:
for layers in range(1, 7):
    print(layers, round(100 * 0.25 ** layers, 3))

After six layers, less than three hundredths of one percent of the learning signal is left, and Figure 4.3 shows how fast it disappears. This is called the **vanishing gradient** problem, and it is exactly why deep sigmoid networks of the 2000s learned so slowly in their early layers. **When every layer shrinks the learning signal, the early layers of a deep network hear almost nothing and barely learn.**

> 🖼️ **Figure 4.3:** The share of the learning signal that survives sigmoid layers, even when every neuron sits at its steepest point. By the third layer the bars are already too small to see.

## The ReLU and Why It Won

The ReLU is about as simple as an activation can be. If the weighted sum is positive, the neuron passes it on unchanged, and if it is zero or negative, the neuron outputs zero. It works like a ticket barrier that stops everyone without a ticket and lets everyone with one walk straight through at full speed. In code it is a single numpy function.

In [ ]:
def relu(z):
    return np.maximum(0, z)

print(relu(z))
print((z > 0).astype(int))

The second line is the slope. For every positive input the slope is exactly 1, so a small change passes through a working ReLU neuron at full strength, however large the input is. There is no flat edge on the positive side for a neuron to get stuck on, and a chain of working ReLU layers passes the learning signal backwards without shrinking it. It is also very cheap to compute, since it needs no exponentials at all, which matters when a network has millions of neurons.

You can now watch all three bends on the same deep network. The cell trains the three layer network on the plants with each activation for exactly 600 rounds through the training data, and prints the final loss, a number you will meet properly in Chapter 5, where lower means fewer mistakes. Scikit learn calls the sigmoid `"logistic"`, and the setting `n_iter_no_change=600` stops it from giving up early when progress is slow.

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # hide the note that training stopped at 600 rounds

for act in ["logistic", "tanh", "relu"]:
    net = MLPClassifier(hidden_layer_sizes=(8, 8, 8),
                        activation=act, max_iter=600,
                        n_iter_no_change=600, random_state=7)
    net.fit(X_train, y_train)
    print(act, round(net.loss_curve_[-1], 3),
          round(net.score(X_test, y_test), 2))

After 600 rounds the sigmoid network has barely learned anything, while the tanh and ReLU networks are both perfect. Figure 4.4 shows the loss after every round, and the difference is striking. The sigmoid's line hardly moves from where it started, which is exactly the vanishing gradient at work, while ReLU is the first to drop below 0.1, after about 200 rounds.

> 🖼️ **Figure 4.4:** The loss of the same three layer network after every round of training, measured with three different activations. The sigmoid network is still stuck after 600 rounds, while tanh and ReLU both learn the circle.

ReLU has one weakness worth knowing about. If a neuron's weighted sum ends up negative for every example it sees, its output is always zero and its slope is always zero too, so it stops learning for good. This is called a **dead ReLU**. A popular fix is the **leaky ReLU**, which lets a tiny fraction of negative numbers leak through, usually one hundredth, so the slope on the negative side is small but never zero.

In [ ]:
def leaky_relu(z):
    return np.where(z > 0, z, 0.01 * z)

print(leaky_relu(z))

Figure 4.5 draws ReLU and its leaky cousin side by side. **ReLU won because its slope is a full 1 for every positive input, so the learning signal reaches the early layers of a deep network without fading away, and it costs almost nothing to compute.**

> 🖼️ **Figure 4.5:** ReLU keeps positive numbers and turns negative ones into zero. The leaky ReLU lets a little of each negative number through, so its neurons can never die completely.

## Choosing an Activation

With several bends to choose from, it helps to separate two very different jobs. The activations inside the hidden layers exist to bend the space so the network can learn, and here the choice is mostly settled. The activation on the output layer has a different job, which is to put the network's final answer into the right form for the question being asked.

For hidden layers, start with ReLU. It trains quickly, it works well in almost every kind of network, and it is the default in PyTorch examples you will meet from Chapter 10 onwards. If many of your neurons seem to die, try the leaky ReLU, and if you read about large language models in Book 8, you will meet smoother relatives of ReLU such as GELU, which follow the same basic shape. The sigmoid and tanh are rarely used in hidden layers of ordinary networks today, although the tanh still appears inside some of the sequence models of Chapter 18.

For the output layer, ask what kind of answer you need. Table 4.1 gathers the common cases in one place.

**Table 4.1: A practical guide to choosing activations. Hidden layers bend the space, while the output layer shapes the final answer.**

| Where the neuron is | Activation to use | Why |
|---|---|---|
| Hidden layers | ReLU | Fast to learn, slope 1 for positives |
| Hidden layers, if neurons die | Leaky ReLU | Never has a slope of exactly zero |
| Output, a yes or no answer | Sigmoid | Gives a probability between 0 and 1 |
| Output, one of several classes | Softmax, Chapter 5 | Gives probabilities that add up to 1 |
| Output, a number such as a price | None | The answer may be any size |

The last row deserves a moment. If a network predicts the price of a house, its output must be free to be 85,000 or 3,200,000, so squeezing it through a sigmoid that can only give numbers between 0 and 1 would make the task impossible. In that case the output neuron simply passes on its weighted sum with no activation at all. **Use ReLU in the hidden layers to bend the space, and choose the output activation to match the kind of answer the question needs.** The sigmoid is still the right choice at the end of a yes or no network, because there its job is not to pass learning signals through many layers but to turn one final number into a probability.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Collapse three layers.** Add a third linear layer `W3` of shape `(2, 1)` with its own bias after `W2`, then combine all three into a single `W` and `b` and check that the answer for `x` is unchanged.
2. **Go deeper with the sigmoid.** Train the plants network with `hidden_layer_sizes=(8,)` and then `(8, 8, 8, 8, 8)` using `"logistic"`, with the same 600 rounds, and compare the final losses.
3. **Find a dead neuron.** Build the plants network with ReLU, then use `net.coefs_` and `net.intercepts_` to compute the first hidden layer's outputs for every training example. Count how many of the eight neurons give zero for all of them.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why does a network with no activations collapse into a single layer?**
2. **What is the largest slope of the sigmoid, and why does that matter in a deep network?**
3. **What does it mean for a neuron to be saturated?**
4. **Why did ReLU make deep networks easier to train?**
5. **Which activation would you put on the output of a network that predicts a temperature?**

### Answers

1. Because a chain of multiply and add steps is itself just one multiply and add step, in the same way that two photocopier zooms make one zoom.
2. It is 0.25, so each sigmoid layer passes back at most a quarter of the learning signal, and after a few layers almost nothing is left for the early layers.
3. Its weighted sum is so far from zero that the curve is flat there, so small changes in its input no longer change its output.
4. Its slope is exactly 1 for every positive input, so the learning signal passes back through many layers without fading, and it is very cheap to compute.
5. None at all, because a temperature can be any number, and an activation such as the sigmoid would squeeze it into a narrow range.

## What Comes Next

You now know why every neuron needs a bend, why the sigmoid made deep networks so hard to train, and why ReLU became the default.

- **Chapter 5, Loss Functions,** turns "how wrong is this network" into a single number the network can try to shrink.
- **Chapter 6, Gradient Descent From Scratch,** uses that number to improve a network one small step at a time.

Chapter 5 begins in 1801, with a faint light in the sky that was found, lost behind the sun, and found again by arithmetic.